# RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/yq/wt0bym0d4c31msbvz5mz2xz40000gn/T/ipykernel_858/4052997885.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader


In [2]:
### Read all the pdf's inside the directory
from pathlib import Path

def process_all_pdfs(pdf_directory):
    """Process all pdfs in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    #find all pdfs recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader=PyMuPDFLoader(str(pdf_file))
            documents=loader.load()

            #Add source info to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")
    
    print(f"\n Total Documents loaded : {len(documents)}")
    return all_documents


#Process all PDFs in the data directory
all_pdf_documents=process_all_pdfs("../data/pdf")


Found 2 PDF files to process

Processing: Cybersecurity_Basics.pdf
 Loaded 1 pages

Processing: Cloud_Computing_Notes.pdf
 Loaded 1 pages

 Total Documents loaded : 1


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-24T18:25:32+00:00', 'source': '../data/pdf/Cybersecurity_Basics.pdf', 'file_path': '../data/pdf/Cybersecurity_Basics.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-24T18:25:32+00:00', 'trapped': '', 'modDate': "D:20260624182532+00'00'", 'creationDate': "D:20260624182532+00'00'", 'page': 0, 'source_file': 'Cybersecurity_Basics.pdf', 'file_type': 'pdf'}, page_content='Cybersecurity Basics\nOverview\nCybersecurity focuses on protecting systems, networks, and data from attacks.\nCommon Threats\n• Phishing\n• Malware\n• Ransomware\n• Social Engineering\nBest Practices\nUse strong passwords, enable multi-factor authentication, keep software updated, and verify\nsuspicious links.'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creato

In [10]:
### text splitting get into chunks

def split_documents(documents,chunk_size=100,chunk_pverlap=20):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_pverlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )

    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of chunk
    if split_docs:
        print(f"\nExample chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [11]:
chunks=split_documents(all_pdf_documents)
chunks

Split 2 documents into 10 chunks

Example chunk: 
Content: Cybersecurity Basics
Overview...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-24T18:25:32+00:00', 'source': '../data/pdf/Cybersecurity_Basics.pdf', 'file_path': '../data/pdf/Cybersecurity_Basics.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-24T18:25:32+00:00', 'trapped': '', 'modDate': "D:20260624182532+00'00'", 'creationDate': "D:20260624182532+00'00'", 'page': 0, 'source_file': 'Cybersecurity_Basics.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-24T18:25:32+00:00', 'source': '../data/pdf/Cybersecurity_Basics.pdf', 'file_path': '../data/pdf/Cybersecurity_Basics.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-06-24T18:25:32+00:00', 'trapped': '', 'modDate': "D:20260624182532+00'00'", 'creationDate': "D:20260624182532+00'00'", 'page': 0, 'source_file': 'Cybersecurity_Basics.pdf', 'file_type': 'pdf'}, page_content='Cybersecurity Basics\nOverview'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-06-24T18:25:32+00:00', 'source': '../data/pdf/Cybersecurity_Basics.pdf', 'file_path': '../data/pdf/Cybersecurity_Basics.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)

### Embedding and VectorStore db

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/rithik/Downloads/AI/RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. "
                f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            NumPy array of embeddings with shape
            (len(texts), embedding_dimension).
        """
        if self.model is None:
            raise ValueError("Model not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True,
        )
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# Initialize the embedding manager
embedding_manager = EmbeddingManager()
EmbeddingManager

# # Example usage
# texts = [
#     "Machine learning is fascinating.",
#     "Sentence transformers generate embeddings.",
# ]

# embeddings = embedding_manager.generate_embeddings(texts)
# print(embeddings.shape)

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9319.67it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/yq/wt0bym0d4c31msbvz5mz2xz40000gn/T/ipykernel_858/4217184443.py:22: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Embedding dimension: {self.model.get_sentence_embedding_dimension()}"


__main__.EmbeddingManager